In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pickle
import time
import seaborn as sns
from scipy.optimize import fmin, minimize, LinearConstraint, Bounds
from efficient_fpt.multi_stage_cy import compute_loss_parallel, print_num_threads
import pandas as pd 

real_data_m_1 = pd.read_csv("/users/azhan378/data/azhang/efficient-fpt/data/behav_subj1.csv")


In [3]:
real_data_m_1 

,trialNum,firstVal,secondVal,firstIsChosen,decisionRT,firstFixLat,secondFixLat,thirdFixLat
0,6,1,3,0,0.827066,0.162,0.510,NaN
1,11,1,2,1,0.847049,0.164,0.372,0.616
2,18,1,5,0,0.897351,0.166,0.414,NaN
3,61,1,1,0,0.877376,0.163,0.415,NaN
4,62,1,5,0,0.937264,0.178,0.430,NaN
...,...,...,...,...,...,...,...,...
582,481,5,5,1,0.739332,0.213,0.441,0.617
583,494,5,2,1,0.807278,0.222,0.526,0.670
584,503,5,5,1,0.757297,0.184,0.452,0.632
585,518,5,3,1,0.687202,0.160,0.404,0.540


In [ ]:
# data preprocessing 



In [4]:
# Define priors for params 
from scipy.stats import beta, gamma, lognorm


def log_prior(z):
    M = float(max(rt_data))
    eta, kappa, a, b, x0 = z
    # --- hard constraints ---
    if not (0.0 < eta < 1.0):
        return -np.inf
    if not (kappa > 0.0 and a > 0.0):
        return -np.inf
    if not (0.0 < b < a / M):
        return -np.inf
    if not (-a < x0 < a):
        return -np.inf
    log_eta_prior = beta.logpdf(eta, a=2, b=2)
    log_kappa_prior = gamma.logpdf(kappa, a=2, scale=1/4.0) # lognorm.logpdf(kappa, s=1, scale=np.exp(0))
    log_a_prior = gamma.logpdf(a, a=2, scale=1/1.0) # lognorm.logpdf(a, s=1, scale=np.exp(0))
    log_b_prior = beta.logpdf(b, a=2, b=2, loc=0, scale=a / np.max(rt_data))
    log_x0_prior = beta.logpdf(x0, a=2, b=2, loc=-a, scale=2 * a)
    return log_eta_prior + log_kappa_prior + log_a_prior + log_b_prior + log_x0_prior